In [ ]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module

# Reload the pipeline module so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
get_snapshot_xml_file_path = ncbi_snapshot_module.get_snapshot_xml_file_path
resolve_ncbi_protein_xml_snapshot = (
    ncbi_snapshot_module.resolve_ncbi_protein_xml_snapshot
)

from src.pago_pipeline.storage import sha256_of_file, sha256_of_lines

In [ ]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================
dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your NCBI email and optional API key at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent
NCBI_EMAIL = os.getenv("NCBI_EMAIL")
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

if not NCBI_EMAIL:
    raise ValueError(
        "NCBI_EMAIL was not found in the environment. "
        "Please define it in your .env file."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"NCBI email configured: {bool(NCBI_EMAIL)}")
print(f"NCBI API key configured: {bool(NCBI_API_KEY)}")

In [ ]:
# =============================================================================
# CELL 3 — Define XML snapshot configuration
# =============================================================================

# Notebook 01 expects notebook 00 to have already materialized the upstream
# protein UID snapshot in the source snapshot root directory below.
SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_uid_snapshots"
)
XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)

XML_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
MAX_RETRY_ATTEMPTS = 5

# EFetch paging is capped at 10,000 identifiers per request. 100 keeps each
# response near 0.9 MB, which limits how long a single body is exposed to a
# mid-stream drop. Raising it is a measured trial, not a default: see
# documentation/ncbi_retrieval_performance_implementation.md.
XML_BATCH_SIZE = 100
XML_REQUEST_DELAY_SECONDS = None

# Bounded concurrency. Keep this at 1 while measuring the batch-size change on
# its own; raise to 2 first, and only to 4 once a trial shows idle latency and
# zero HTTP 429 responses. NCBI's allowance covers every request issued under
# the same API key, not just this process.
XML_MAX_CONCURRENT_REQUESTS = 4
XML_MAX_REQUEST_STARTS_PER_SECOND = None  # None -> conservative default

# Reuse one keep-alive HTTPS connection per worker instead of paying a TCP and
# TLS handshake per request. Hold this fixed while measuring anything else.
XML_REUSE_HTTP_CONNECTION = False

# A failed run keeps its validated batches in the workspace so that the rerun
# re-fetches only what is missing. The workspace is removed after a successful
# publication.
ENABLE_BATCH_RESUME = True
PURGE_BATCH_WORKSPACE_ON_SUCCESS = True

UPDATE_LATEST_DIRECTORY = True

print(f"Source UID snapshot root directory: {SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY}")
print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"XML snapshot mode: {XML_SNAPSHOT_MODE}")
print(f"XML batch size: {XML_BATCH_SIZE}")
print(f"XML max concurrent requests: {XML_MAX_CONCURRENT_REQUESTS}")
print(f"XML HTTP connection reuse: {XML_REUSE_HTTP_CONNECTION}")
print(f"Batch resume enabled: {ENABLE_BATCH_RESUME}")
print("Expected upstream UID snapshot: latest snapshot produced by notebook 00")

In [ ]:
# =============================================================================
# CELL 4 — Resolve active XML snapshot
# =============================================================================

try:
    xml_snapshot_payload = resolve_ncbi_protein_xml_snapshot(
        snapshot_mode=XML_SNAPSHOT_MODE,
        snapshot_root_directory=XML_SNAPSHOT_ROOT_DIRECTORY,
        source_uid_snapshot_root_directory=SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY,
        xml_batch_size=XML_BATCH_SIZE,
        max_retry_attempts=MAX_RETRY_ATTEMPTS,
        xml_request_delay_seconds=XML_REQUEST_DELAY_SECONDS,
        max_concurrent_requests=XML_MAX_CONCURRENT_REQUESTS,
        max_request_starts_per_second=XML_MAX_REQUEST_STARTS_PER_SECOND,
        reuse_http_connection=XML_REUSE_HTTP_CONNECTION,
        enable_batch_resume=ENABLE_BATCH_RESUME,
        purge_batch_workspace_on_success=PURGE_BATCH_WORKSPACE_ON_SUCCESS,
        ncbi_email=NCBI_EMAIL,
        ncbi_api_key=NCBI_API_KEY,
        update_latest_directory=UPDATE_LATEST_DIRECTORY,
    )
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "No upstream protein UID snapshot was found. "
        "Run notebook 00_ncbi_protein_ids_query.ipynb first, "
        "then rerun notebook 01_ncbi_protein_xml_snapshot.ipynb."
    ) from exc

xml_snapshot_directory = xml_snapshot_payload["snapshot_directory"]
manifest_file_path = xml_snapshot_payload["manifest_file_path"]
protein_uids_file_path = xml_snapshot_payload["protein_uids_file_path"]
xml_snapshot_manifest = xml_snapshot_payload["manifest"]
protein_uids = xml_snapshot_payload["protein_uids"]

xml_file_path = get_snapshot_xml_file_path(
    snapshot_directory=xml_snapshot_directory,
)
uid_dataset_sha256 = sha256_of_lines(
    text_lines=protein_uids,
    deduplicate_lines_preserving_order=False,
    sort_lines=False,
)

print(f"Resolved XML snapshot directory: {xml_snapshot_directory}")
print(f"Resolved XML UID count: {len(protein_uids)}")
print(f"Resolved XML batch count: {xml_snapshot_manifest['batch_count']}")

In [ ]:
# =============================================================================
# CELL 5 — Print XML snapshot summary
# =============================================================================

# The consolidated SHA-256 is accumulated while the snapshot is written and is
# revalidated whenever a saved snapshot is reloaded, so this notebook reads it
# from the manifest instead of streaming several hundred megabytes through the
# hash function on every run. Set the flag below to force an independent check.
VERIFY_XML_FILE_SHA256_BY_REHASHING = False

manifest_xml_file_sha256 = xml_snapshot_manifest["xml_file_sha256"]

if VERIFY_XML_FILE_SHA256_BY_REHASHING:
    xml_file_sha256 = sha256_of_file(input_file_path=xml_file_path)
    xml_file_sha256_source = "recomputed from the saved file"
else:
    xml_file_sha256 = manifest_xml_file_sha256
    xml_file_sha256_source = "manifest, validated when the file was written"

print("XML snapshot resolved successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Consolidated XML SHA-256: {xml_file_sha256}")
print(f"SHA-256 source: {xml_file_sha256_source}")
print(f"Total protein UIDs covered: {len(protein_uids)}")
print(f"Total XML batches used for consolidation: {xml_snapshot_manifest['batch_count']}")

In [ ]:
# =============================================================================
# CELL 6 — Print manifest-derived retrieval summary
# =============================================================================

print(f"Retrieved at UTC: {xml_snapshot_manifest['retrieved_at_utc']}")
print(f"Manifest snapshot format version: {xml_snapshot_manifest['snapshot_format_version']}")
print(f"Manifest retmode: {xml_snapshot_manifest['retmode']}")
print(f"Manifest rettype: {xml_snapshot_manifest.get('rettype')}")
print(f"Manifest batch size: {xml_snapshot_manifest['batch_size']}")
print(f"Manifest batch count: {xml_snapshot_manifest['batch_count']}")
print(f"Manifest reused batch count: {xml_snapshot_manifest.get('reused_batch_count')}")
print(f"Manifest fetched batch count: {xml_snapshot_manifest.get('fetched_batch_count')}")
print(
    "Manifest normalized protein UID count: "
    f"{xml_snapshot_manifest['normalized_protein_uid_count']}"
)
print(
    "Manifest consolidated record count: "
    f"{xml_snapshot_manifest['consolidated_record_count']}"
)
print(
    "Source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)

request_policy = xml_snapshot_manifest.get("request_policy")
if request_policy:
    print()
    print("=== REQUEST POLICY ===")
    for policy_name, policy_value in sorted(request_policy.items()):
        print(f"{policy_name}: {policy_value}")

# The payload telemetry covers the whole run, including the publication phases
# that happen after the manifest itself is written.
retrieval_telemetry = xml_snapshot_payload.get(
    "retrieval_telemetry",
    xml_snapshot_manifest.get("retrieval_telemetry"),
)
if retrieval_telemetry:
    print()
    print("=== RETRIEVAL TELEMETRY ===")
    for stage_name, stage_summary in retrieval_telemetry["stages"].items():
        print(
            f"{stage_name}: {stage_summary['request_count']} requests, "
            f"{stage_summary['retry_count']} retries, "
            f"{stage_summary['response_byte_total']} bytes, "
            f"wall {stage_summary['wall_seconds_total']} s"
        )
        print(
            f"    latency  median {stage_summary['total_latency']['median_seconds']} s, "
            f"p95 {stage_summary['total_latency']['p95_seconds']} s, "
            f"max {stage_summary['total_latency']['max_seconds']} s"
        )
        print(
            "    time to first byte median "
            f"{stage_summary['time_to_first_byte_latency']['median_seconds']} s, "
            "response read median "
            f"{stage_summary['response_read_latency']['median_seconds']} s"
        )
        print(f"    regulatory sleep {stage_summary['sleep_seconds_total']} s")
        print(f"    failures {stage_summary['failure_counts']}")

    print(f"local phases: {retrieval_telemetry['local_phase_seconds']}")
    print(f"process: {retrieval_telemetry['process']}")
else:
    print()
    print("Retrieval telemetry: not recorded (snapshot predates instrumentation).")

In [ ]:
# =============================================================================
# CELL 7 — Print persisted file hashes
# =============================================================================

saved_manifest_sha256 = sha256_of_file(input_file_path=manifest_file_path)
saved_protein_uids_file_sha256 = sha256_of_file(
    input_file_path=protein_uids_file_path,
)

print("Persisted XML snapshot file hashes:")
print(f"Manifest SHA-256: {saved_manifest_sha256}")
print(f"Protein UIDs file SHA-256: {saved_protein_uids_file_sha256}")
print(f"Consolidated XML file SHA-256: {xml_file_sha256}")

In [ ]:
# =============================================================================
# CELL 8 — Print persisted manifest summary
# =============================================================================

print("Persisted XML snapshot metadata:")
print(f"Manifest path: {manifest_file_path}")
print(f"Protein UIDs file path: {protein_uids_file_path}")
print(f"Manifest XML file name: {xml_snapshot_manifest['xml_file_name']}")
print(f"Manifest XML SHA-256: {xml_snapshot_manifest['xml_file_sha256']}")
print(
    "Manifest immutable snapshot relative path: "
    f"{xml_snapshot_manifest['immutable_snapshot_relative_path']}"
)
print(
    "Manifest source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)

In [ ]:
# =============================================================================
# CELL 9 — Inspect first saved batch records
# =============================================================================

first_three_saved_batch_records = xml_snapshot_manifest["batches"][:3]

for saved_batch_record in first_three_saved_batch_records:
    print(f"Batch index: {saved_batch_record['batch_index']}")
    print(
        "Batch UID interval: "
        f"{saved_batch_record['batch_start_index']}"
        f"..{saved_batch_record['batch_end_index']}"
    )
    print(saved_batch_record["xml_payload_sha256"])
    print(saved_batch_record["protein_uid_count"])
    print("---")

In [ ]:
# =============================================================================
# CELL 10 — Validate persisted snapshot consistency
# =============================================================================

print("XML snapshot resolution completed successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Protein UIDs file: {protein_uids_file_path}")
print(f"Manifest file: {manifest_file_path}")
print(
    "Manifest XML file name matches saved path: "
    f"{xml_snapshot_manifest['xml_file_name'] == xml_file_path.name}"
)

if VERIFY_XML_FILE_SHA256_BY_REHASHING:
    print(
        "Manifest XML SHA-256 matches the rehashed XML file: "
        f"{manifest_xml_file_sha256 == xml_file_sha256}"
    )
else:
    print(
        "Manifest XML SHA-256 was accepted without rehashing; it was computed "
        "while the file was written and is re-verified on every snapshot reuse. "
        "Set VERIFY_XML_FILE_SHA256_BY_REHASHING=True in CELL 5 to check it here."
    )

print(
    "Manifest UID SHA-256 matches resolved UID list: "
    f"{xml_snapshot_manifest['protein_uids_sha256'] == uid_dataset_sha256}"
)
print(
    "Manifest UID count matches resolved UID list: "
    f"{xml_snapshot_manifest['normalized_protein_uid_count'] == len(protein_uids)}"
)

In [ ]:
# =============================================================================
# CELL 11 — Print source UID snapshot provenance
# =============================================================================
print("Source UID snapshot provenance:")
print(
    "Source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)
print(
    "Source UID snapshot directory name: "
    f"{xml_snapshot_manifest['source_uid_snapshot_directory_name']}"
)
print(
    "Source UID snapshot manifest SHA-256: "
    f"{xml_snapshot_manifest['source_uid_snapshot_manifest_sha256']}"
)
print(f"Source UID count: {xml_snapshot_manifest['source_uid_count']}")